# 04 · Tools

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama.

Tools let agents act and look things up. CrewAI offers two ways to write them: a
quick `@tool` **function**, or a `BaseTool` **class** for tools with
configuration or state. This notebook covers both, then the behaviours that
decide reliability: what happens when a tool fails, how the result can *be* the
answer, caching, and usage limits. It ends with CrewAI's ready-made tool library.

**You will learn**
- `@tool` functions and `BaseTool` classes, and what the model sees of each
- Tool failures: reported to the agent, **recorded** in the output, or raised
- `result_as_answer`: the tool's output becomes the task's final answer
- Caching tool results, and limiting how often a tool can be used
- The `crewai_tools` library

**Prerequisites:** notebooks 01–02.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew
from crewai.tools import tool, BaseTool

llm = get_llm()

## 1 · Why tools, and how agents call them

An agent without tools can only write text from what the model already knows.
With tools it can check facts, compute, and act. When an agent has tools, CrewAI
adds their names, descriptions and argument schemas to the prompt (or sends them
as native tool definitions when the model supports function calling), and runs
whatever the model requests:

```
 agent's turn ──► model asks for tool X(args) ──► CrewAI runs X ──► result back to the model ──► … ──► final answer
```

As in every framework: the model reads only the tool's **name, description and
argument schema**, so write those for the model.

## 2 · Writing tools

**`@tool` functions** are the quick way. The decorator's argument is the tool's
name, the docstring is its description, and the type hints are its arguments:

In [ ]:
@tool('Weather lookup')
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use it for any question about weather or packing."""
    fake = {'pune': '31°C, humid', 'oslo': '4°C, rain', 'lima': '19°C, cloudy'}
    return f"{city}: {fake.get(city.lower(), '20°C, clear')}"

print('name       :', get_weather.name)
print('description:', get_weather.description)
print('arguments  :', get_weather.args_schema.model_json_schema()['properties'])

**`BaseTool` classes** are for tools that need configuration, state, or a
precise argument schema. You subclass it, set `name`, `description` and an
`args_schema` (a Pydantic model), and implement `_run`:

In [ ]:
class ConvertInput(BaseModel):
    amount: float = Field(description='Amount in euros')
    currency: str = Field(description='Target currency code: USD or INR')


class CurrencyConverter(BaseTool):
    name: str = 'Currency converter'
    description: str = 'Convert an amount in EUR to USD or INR at the configured rates.'
    args_schema: type[BaseModel] = ConvertInput
    rates: dict = {'USD': 1.08, 'INR': 90.5}              # configuration lives on the instance

    def _run(self, amount: float, currency: str) -> str:
        rate = self.rates.get(currency.upper())
        if rate is None:
            return f'Unsupported currency {currency}. Use USD or INR.'
        return f'{amount} EUR = {amount * rate:.2f} {currency.upper()}'


converter = CurrencyConverter(rates={'USD': 1.10, 'INR': 91.0})   # a different configuration per instance

travel = Agent(role='Travel assistant', goal='Answer travel questions with facts',
               backstory='You always use tools for weather and money.', llm=llm,
               tools=[get_weather, converter])
task = Task(description='{question}', expected_output='A short, factual answer.', agent=travel)
out = await Crew(agents=[travel], tasks=[task]).kickoff_async(
    inputs={'question': 'Should I pack an umbrella for Oslo, and what is 200 EUR in INR?'})
print(out.raw)

| Use | When |
|---|---|
| `@tool` function | simple, stateless tools: most tools |
| `BaseTool` class | configuration per instance, shared clients, a precise schema, or **checkpointed crews** (a class tool can be saved in a checkpoint; notebook 09 shows why a function tool breaks crew checkpointing) |

## 3 · When a tool fails

A tool raises an exception. What happens?

In [ ]:
@tool('Divide')
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

calc = Agent(role='Calculator', goal='Compute with tools', backstory='You always use tools.', llm=llm, tools=[divide])
calc_task = Task(description='Use the Divide tool to compute 10 / 0 and report what happened.',
                 expected_output='The result or the problem.', agent=calc)
out = await Crew(agents=[calc], tasks=[calc_task]).kickoff_async()

print('answer            :', out.raw[:150])
print('has_tool_failures :', out.has_tool_failures)
for record in out.tasks_output[0].tool_failures:
    print('recorded failure  :', record.tool_name, record.tool_args, '→', record.failure.code,
          f'({record.failure.message}), retryable={record.failure.retryable}')

By default CrewAI does two things:

1. **Reports the error to the agent** as the tool result, so the model can react
   (as the OpenAI Agents SDK does by default, and unlike LangGraph).
2. **Records the failure** in the output: `has_tool_failures` and each task's
   `tool_failures` tell your code exactly which tool failed, with which arguments
   and why, even when the agent's answer glosses over it. (Note: it's a property, not a method.) That's the raw material
   for "did it really work?" checks.

`tool_failure_policy` (on a tool, agent, task or crew) changes this:

| Policy | Effect |
|---|---|
| default | record the failure, emit an event, keep going |
| `'ignore'` | don't record or act on failures (older behaviour) |
| `'raise'` | record, then abort the task with `ToolExecutionFailedError` |

Use `'raise'` for tools whose failure makes the rest of the task meaningless.

## 4 · `result_as_answer`: the tool's output *is* the answer

Sometimes the tool's output must reach the user **unchanged**: an official
price, a legal text, a computed figure. `result_as_answer=True` ends the task
with the tool's output as the final answer, so the model gets no chance to
reword it:

In [ ]:
@tool('Official price', result_as_answer=True)
def official_price(product: str) -> str:
    """The official price of a product, exactly as it must be quoted."""
    return f'{product}: EUR 1,299.00 incl. VAT (price list 2026-09)'

quoter = Agent(role='Sales assistant', goal='Quote prices', backstory='Uses the price tool.',
               llm=llm, tools=[official_price])
out = await Crew(agents=[quoter], tasks=[Task(description='How much is the robot kit?',
                                              expected_output='The price', agent=quoter)]).kickoff_async()
print(repr(out.raw))

## 5 · Caching and usage limits

**Caching.** CrewAI can cache tool results: when an agent calls a tool with
exactly the same arguments again, it can reuse the stored result instead of
running the tool. Caching is on by default for agents and crews (`cache=True`),
and a tool can decide *which* results may be cached with a `cache_function`.
Let's check it with two tasks that need the same lookup:

In [ ]:
lookups = []

@tool('Stock lookup')
def stock(sku: str) -> str:
    """Look up the warehouse stock for a product SKU."""
    lookups.append(sku)
    return f'{sku}: 12 in stock'

clerk = Agent(role='Warehouse clerk', goal='Answer stock questions', backstory='Always uses the tool.',
              llm=llm, tools=[stock])
t1 = Task(description='How many units of SKU C2-STD are in stock?', expected_output='a number', agent=clerk)
t2 = Task(description='Check again: how many units of SKU C2-STD are in stock?', expected_output='a number', agent=clerk)
await Crew(agents=[clerk], tasks=[t1, t2]).kickoff_async()
print('tool body actually ran', len(lookups), 'time(s) for', len(lookups) and 'identical calls:', lookups)

**What we observed:** the tool body ran **twice** for two identical calls, even
though they came from two *separate, sequential* tasks. In our testing (CrewAI
1.15, native tool calling on Ollama) the cache never prevented a repeat. It may
behave differently on other models or code paths, so check your own run. The
lesson holds either way: **don't rely on caching to stop duplicate work or
duplicate side effects**. At best it's a cost optimisation for read-only tools. For tools that *act* (send, pay,
write), turn caching off for that tool (`cache_function=lambda args, result: False`)
so a repeated request is never silently skipped, and make the tool idempotent.

**Usage limits.** `max_usage_count` on a tool caps how many times it can run
(per tool instance), which is a simple guard against loops that hammer an
expensive API:

```python
search = MySearchTool(max_usage_count=3)     # a fourth call is refused
```

## 6 · Ready-made tools: `crewai_tools`

The `crewai-tools` package has dozens of tools: reading files and directories,
scraping websites, searching the web (with an API key), querying databases, RAG
over PDFs and more. A local example, reading one of the Nimbus documents:

In [ ]:
from crewai_tools import FileReadTool
from common import SECTION_DIR

reader = FileReadTool(file_path=str(SECTION_DIR / 'data' / 'kb' / '05_permits_and_regulation.md'))
analyst = Agent(role='Regulation analyst', goal='Answer from documents', backstory='Reads before answering.',
                llm=llm, tools=[reader])
out = await Crew(agents=[analyst], tasks=[Task(
    description='What condition did Utrecht attach to its sidewalk permit? Read the file first.',
    expected_output='The condition, in one sentence.', agent=analyst)]).kickoff_async()
print(out.raw)

Many library tools call external services, so read what each one sends where
before using it, and prefer tools whose data stays local for sensitive
documents.

## 7 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Vague tool names / docstrings | wrong tool, or none | clear names, "use it for…" descriptions |
| Trusting the answer when a tool failed | answers that hide errors | check `has_tool_failures` / `tool_failures` |
| Letting the model reword authoritative data | changed prices, legal text | `result_as_answer=True` |
| Relying on the cache to prevent duplicate actions | concurrent duplicates run anyway | no caching for side-effecting tools; idempotency |
| Function tools in checkpointed crews | checkpoints silently fail (notebook 09) | `BaseTool` classes |
| Unlimited calls to an expensive API | runaway cost | `max_usage_count`, agent `max_iter` |

## 8 · Check yourself

<details><summary><b>1.</b> When would you write a <code>BaseTool</code> class instead of a <code>@tool</code> function?</summary>

When the tool needs per-instance configuration or state (rates, clients, credentials), a precise argument schema, or must be serialisable, e.g. for crew checkpoints.
</details>

<details><summary><b>2.</b> A tool raises. What does the agent see by default, and what does your code see?</summary>

The agent receives the error as the tool's result and can react. Your code sees the failure recorded in the output: `has_tool_failures` and each task's `tool_failures` (tool, arguments, error code, message, retryable).
</details>

<details><summary><b>3.</b> What does <code>result_as_answer=True</code> do, and when is it the right choice?</summary>

It makes the tool's output the task's final answer, with no model rewording. Use it for authoritative outputs that must be quoted exactly.
</details>

<details><summary><b>4.</b> Why shouldn't you rely on caching to avoid sending the same email twice?</summary>

Caching is at best an optimisation for identical *read* calls (and in our tests it didn't even prevent repeats). Side-effecting tools should have caching disabled and be idempotent (check "already sent?" themselves).
</details>

## Takeaway

Write quick tools with `@tool` and configurable (or checkpointable) ones as
`BaseTool` classes, with names and descriptions written for the model. Failures
are reported to the agent **and recorded** for your code; choose `'raise'` when a
failure should stop the task. Use `result_as_answer` for authoritative outputs,
treat caching as a read-only optimisation, and cap expensive tools.

Next → `05_processes_and_collaboration.ipynb`: how crews run their tasks, in
sequence, in parallel, or under a manager.